# Fundamentos de *Deep Learning I* 
> Héctor J. Hortúa, PhD · Instituto de Neurociencias -SavIA-Lab
## Carga de datos en **PyTorch**: `Dataset`, `DataLoader`, `batch` y `collate_fn`

>  **Módulo 2 — Herramientas y manejo de datos**
> **Framework:** PyTorch

Una red neuronal no se entrena con *todo* el conjunto de datos a la vez, sino con **pequeños lotes** (*batches*) que se le van entregando uno tras otro. Preparar esos lotes —leerlos, barajarlos, agruparlos, transformarlos y hacerlo rápido— es el trabajo del **pipeline de datos**. En PyTorch ese pipeline se apoya en dos piezas con responsabilidades bien separadas:

- **`Dataset`** — responde a la pregunta *"¿qué es una muestra y cómo la obtengo?"*.
- **`DataLoader`** — responde a *"¿cómo agrupo muestras en lotes, las barajo y las cargo en paralelo?"*.

En el medio aparece la pieza que más confunde y que este notebook explica con calma: **`collate_fn`**, la función que decide *cómo se combinan varias muestras en un lote*.

### Objetivos

1. Escribir un `Dataset` propio (`__len__` y `__getitem__`) y usar los de conveniencia.
2. Distinguir `Dataset` **map-style** de **iterable-style**.
3. Configurar un `DataLoader` (batch, shuffle, drop_last, workers).
4. Entender qué hace el **collate por defecto** y escribir un **`collate_fn`** propio: primero con imágenes, luego el caso clásico de **secuencias de largo variable con *padding***.
5. Aplicar **transformaciones** a imágenes (`torchvision.transforms`) y a texto.

> Todos los ejemplos usan **datos sintéticos**: corren sin descargar nada.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, IterableDataset
import numpy as np
import pandas as pd

torch.manual_seed(0)
print("PyTorch:", torch.__version__)

## 1. `Dataset`: qué es una muestra

Un `Dataset` **map-style** es cualquier objeto que implementa dos métodos:

- **`__len__(self)`** — cuántas muestras hay.
- **`__getitem__(self, i)`** — devuelve la muestra número `i` (típicamente una tupla `(entrada, etiqueta)`).

Se llama *map-style* porque funciona como un mapa `índice → muestra`: PyTorch puede pedir la muestra `i` en cualquier orden. Nada más. Con esos dos métodos, el `DataLoader` ya sabe recorrerlo.

In [ ]:
class DatasetSintetico(Dataset):
    def __init__(self, n=100, dim=4):
        self.X = torch.randn(n, dim)
        self.y = (self.X.sum(dim=1) > 0).long() 

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

ds = DatasetSintetico(n=100, dim=4)
print("Tamaño del dataset:", len(ds))
x0, y0 = ds[0]                      # esto llama a __getitem__(0)
print("Muestra 0 -> x:", x0, "| y:", y0.item())

### Un atajo: `TensorDataset`

Cuando tus datos ya son tensores, no necesitas escribir la clase: `TensorDataset` la genera por ti emparejando tensores por su primera dimensión.

In [ ]:
X = torch.randn(100, 4)
y = (X.sum(dim=1) > 0).long()
ds_conveniencia = TensorDataset(X, y)
print("Muestra 5:", ds_conveniencia[5])

### Map-style vs Iterable-style

- **Map-style** (el de arriba): sabes cuántas muestras hay y puedes acceder por índice. Es el caso habitual (datos en memoria, imágenes en disco numeradas, filas de un CSV).
- **Iterable-style** (`IterableDataset`): defines `__iter__` y las muestras **fluyen** una tras otra, sin índice ni longitud conocida. Es para **streams**: leer de un socket, de un archivo gigantesco que no cabe en memoria, o de un generador infinito.

La regla práctica: si puedes indexar, usa map-style. Reserva `IterableDataset` para flujos que no se pueden indexar.

In [ ]:
class StreamSintetico(IterableDataset):
    def __init__(self, n=5):
        self.n = n
    def __iter__(self):
        # genera muestras al vuelo, como si llegaran de un stream
        for _ in range(self.n):
            x = torch.randn(3)
            yield x, x.sum()

for muestra in StreamSintetico(n=3):
    print(muestra)

## 2. `DataLoader`: de muestras a lotes

El `Dataset` entrega **una** muestra; el `DataLoader` las **agrupa en lotes**, las **baraja** y puede **cargarlas en paralelo**. Sus argumentos más importantes:

| Argumento | Qué hace |
|---|---|
| `batch_size` | cuántas muestras por lote |
| `shuffle=True` | baraja el orden en cada época (esencial al entrenar) |
| `drop_last=True` | descarta el último lote si queda incompleto |
| `num_workers` | procesos paralelos que precargan datos (0 = en el proceso principal) |
| `collate_fn` | **cómo** combinar las muestras del lote (lo vemos en la sección 3) |

> **Aviso en Windows / notebooks:** usa `num_workers=0`. Con `num_workers>0` en Windows, el arranque de subprocesos exige proteger el código con `if __name__ == '__main__':`, lo que da problemas dentro de notebooks. En Linux/servidores sí conviene subirlo.

In [ ]:
loader = DataLoader(ds, batch_size=16, shuffle=True, drop_last=True, num_workers=0)

for i, (xb, yb) in enumerate(loader):
    print(f"lote {i}: xb.shape = {tuple(xb.shape)}, yb.shape = {tuple(yb.shape)}")
    if i == 2:
        break
print("\nCada lote apila 16 muestras: (16, 4) para X y (16,) para y")

Fíjate en lo que pasó: cada muestra individual era `x` de forma `(4,)`, y el `DataLoader` apiló 16 de ellas en un tensor `(16, 4)`, **añadiendo una dimensión de lote al frente**. Esa operación de "apilar" es exactamente lo que hace el `collate_fn` por defecto. Vamos a ello.

## 3. `batch` y `collate_fn`: cómo se arma un lote

Cuando el `DataLoader` ya tiene la lista de muestras de un lote (por ejemplo `[muestra_0, muestra_1, ..., muestra_15]`), necesita **fusionarlas en tensores**. Ese trabajo lo hace una función llamada **`collate_fn`**. Si no le pasas ninguna, PyTorch usa `default_collate`.

### Qué hace el collate por defecto

`default_collate` toma una lista de muestras y, para cada componente, **apila** los tensores a lo largo de una nueva dimensión de lote. Es decir, convierte una *lista de tuplas* en una *tupla de tensores por lotes*.

In [ ]:
from torch.utils.data import default_collate

# Simulamos lo que recibe el collate: una lista de 3 muestras (x, y)
muestras = [ds[0], ds[1], ds[2]]
print("Entrada (lista de 3 tuplas):")
for m in muestras: print("  ", m[0].shape, m[1])

xb, yb = default_collate(muestras)
print("\nSalida del collate por defecto:")
print("  xb:", xb.shape, "| yb:", yb.shape)

### Ejemplo con imágenes: un `collate_fn` propio y sencillo

Supongamos que cada muestra es una **imagen sintética** `(canales, alto, ancho)` y su etiqueta. El collate por defecto ya las apila en `(lote, canales, alto, ancho)` sin problema. Pero a veces queremos que el lote tenga **otra estructura** —por ejemplo, devolverlo como un **diccionario** con nombres, algo muy común en proyectos reales—. Para eso escribimos nuestro propio `collate_fn`: recibe la lista de muestras y devuelve lo que queramos.

In [ ]:
class ImagenesSinteticas(Dataset):
    def __init__(self, n=20, c=3, h=8, w=8):
        self.imgs = torch.randn(n, c, h, w)          # n imágenes (c,h,w)
        self.labels = torch.randint(0, 10, (n,))     # etiqueta 0..9
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.imgs[i], self.labels[i]

img_ds = ImagenesSinteticas()
print("Una imagen:", img_ds[0][0].shape, "| etiqueta:", img_ds[0][1].item())

In [ ]:
def collate_dict(batch):
    # batch es una lista de (imagen, etiqueta)
    imgs = torch.stack([b[0] for b in batch])       # (lote, c, h, w)
    labels = torch.tensor([b[1] for b in batch])    # (lote,)
    return {"imagenes": imgs, "etiquetas": labels}  # devolvemos un dict con nombres

img_loader = DataLoader(img_ds, batch_size=4, collate_fn=collate_dict)
lote = next(iter(img_loader))
print("Claves del lote:", list(lote.keys()))
print("imagenes:", lote["imagenes"].shape, "| etiquetas:", lote["etiquetas"].shape)

Hasta aquí `collate_fn` parece un lujo: el default ya apilaba bien las imágenes. **El caso donde se vuelve imprescindible es el siguiente.**

### El caso que rompe el default: secuencias de largo variable

En texto (y en muchas series), cada muestra tiene **distinta longitud**: una frase puede tener 3 palabras y otra 10. El collate por defecto intenta apilar tensores de tamaños distintos y **falla**, porque no se pueden apilar en un rectángulo tensores de formas incompatibles. Veámoslo.

In [ ]:
# Tres "frases" ya convertidas a IDs de palabra, de distinta longitud
secuencias = [torch.tensor([5, 2, 7, 1]),
              torch.tensor([3, 9]),
              torch.tensor([4, 8, 6])]

try:
    default_collate(secuencias)          # intentará apilar (4,), (2,), (3,)
except RuntimeError as e:
    print("Error esperado del default_collate:")
    print(" ", e)

La solución es el **padding**: rellenar las secuencias cortas con un valor especial (normalmente `0`, el token `<pad>`) hasta igualar todas a la longitud de la más larga del lote. Así el lote **sí** es un rectángulo `(lote, longitud_máxima)`.

PyTorch trae `torch.nn.utils.rnn.pad_sequence` para esto. Escribimos un `collate_fn` que:

1. Separa secuencias y etiquetas.
2. Guarda las **longitudes reales** (las necesitaremos en RNNs/LSTM, Módulo 7, para ignorar el relleno).
3. Aplica *padding* a las secuencias.
4. Devuelve todo como tensores.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class DatasetTexto(Dataset):
    def __init__(self):
        # cada muestra: (secuencia de IDs de largo variable, etiqueta)
        self.datos = [
            (torch.tensor([5, 2, 7, 1]), 1),
            (torch.tensor([3, 9]),       0),
            (torch.tensor([4, 8, 6]),    1),
            (torch.tensor([2]),          0),
        ]
    def __len__(self): return len(self.datos)
    def __getitem__(self, i): return self.datos[i]

def collate_padding(batch):
    secuencias = [b[0] for b in batch]
    etiquetas  = torch.tensor([b[1] for b in batch])
    longitudes = torch.tensor([len(s) for s in secuencias])   # largos reales
    # pad_sequence rellena con 0 hasta la longitud máxima del lote
    padded = pad_sequence(secuencias, batch_first=True, padding_value=0)
    return padded, longitudes, etiquetas

txt_loader = DataLoader(DatasetTexto(), batch_size=4, collate_fn=collate_padding)
padded, longitudes, etiquetas = next(iter(txt_loader))
print("Lote con padding (forma", tuple(padded.shape), "):")
print(padded)
print("longitudes reales:", longitudes.tolist())
print("etiquetas       :", etiquetas.tolist())

Observa el resultado: las cuatro secuencias quedaron rellenadas a la longitud de la más larga (4), formando un rectángulo `(4, 4)`. Los ceros son *padding*. Y guardamos `longitudes = [4, 2, 3, 1]` para poder **distinguir el contenido real del relleno**.

Una **máscara** hace explícito qué posiciones son reales (True) y cuáles son relleno (False):

In [ ]:
mascara = padded != 0          # True donde hay token real
print(mascara)
# Estas longitudes/máscara son las que luego alimentan a
# pack_padded_sequence en una RNN para que ignore el relleno (Módulo 7).

> **La idea que hay que llevarse:** el collate por defecto sirve cuando todas las muestras tienen la misma forma. En cuanto tus muestras varían de tamaño —texto, audio, grafos, series de distinta longitud— necesitas un **`collate_fn` propio**, y el patrón casi siempre es *padding + longitudes*.

## 4. Transformaciones

Una **transformación** es una función que se aplica a cada muestra **cuando se lee**, dentro de `__getitem__`. Sirve para normalizar, aumentar datos (*augmentation*), tokenizar texto, etc. El patrón estándar en PyTorch es guardar la transformación en el `Dataset` y aplicarla en `__getitem__`.

### Imágenes: `torchvision.transforms`

`torchvision` ofrece transformaciones listas que se encadenan con `Compose`. Las típicas: `ToTensor` (PIL/ndarray → tensor y escala a [0,1]), `Normalize` (resta media, divide por desviación), y aumentos como `RandomHorizontalFlip` o `RandomRotation`.

> Requiere `pip install torchvision`. Aquí las aplicamos sobre imágenes sintéticas.

In [ ]:
from torchvision import transforms,datasets

# Cadena de transformaciones: aumento + normalización
transformacion = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),   # voltea horizontalmente a veces
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),  # normaliza los 3 canales
])

class ImagenesConTransform(Dataset):
    def __init__(self, n=20, transform=None):
        self.imgs = torch.rand(n, 3, 8, 8)     # imágenes en [0,1]
        self.labels = torch.randint(0, 10, (n,))
        self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = self.imgs[i]
        if self.transform:                     # se aplica al leer la muestra
            img = self.transform(img)
        return img, self.labels[i]

ds_t = ImagenesConTransform(transform=transformacion)
img, lab = ds_t[0]
print("Imagen transformada:", img.shape, "| rango aprox:", (img.min().item(), img.max().item()))

Dos matices importantes sobre las transformaciones de imagen:

- Los **aumentos aleatorios** (flips, rotaciones, recortes) deben aplicarse **solo al conjunto de entrenamiento**, nunca a validación/test: en test queremos evaluar siempre sobre la misma imagen.
- Como se aplican dentro de `__getitem__`, cada época ve una versión ligeramente distinta de cada imagen, lo que **multiplica** la variedad de datos sin ocupar más memoria (lo retomamos en el Módulo 6, visión).

In [ ]:
img_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),                      # -> [0,1], shape (C,H,W)
    transforms.Normalize(mean=[0.5, 0.5, 0.5],  # -> [-1,1]
                         std=[0.5, 0.5, 0.5]),
])

In [ ]:
img_dataset = datasets.ImageFolder(root='../imgs', transform=img_transform)
print("Clases:", img_dataset.classes)    
print("N imágenes:", len(img_dataset))
print("Ejemplo y clase:", img_dataset[0][0].shape, img_dataset[0][1])

In [ ]:
img_loader = DataLoader(img_dataset, batch_size=2, shuffle=True)
x, y = next(iter(img_loader))
print("Batch imágenes:", x.shape, "labels:", y.shape)

In [ ]:

from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, csv_path, window=30, transform=None):
        df = pd.read_csv(csv_path, parse_dates=['Date']).sort_values('Date')
        self.values = df['Monthly Mean Total Sunspot Number'].astype('float32').values
        self.window = window
        self.transform = transform

    def __len__(self):
        # cuántas ventanas deslizantes caben
        return len(self.values) - self.window

    def __getitem__(self, idx):
        window = self.values[idx : idx + self.window]        # entrada
        target = self.values[idx + self.window]              # siguiente valor
        x = torch.tensor(window, dtype=torch.float32).unsqueeze(-1)  # (T,1)
        y = torch.tensor(target, dtype=torch.float32)
        if self.transform:
            x = self.transform(x)
        return x, y

# Normalización sencilla (z-score sobre toda la serie)
class ZScore:
    def __init__(self, mean, std):
        self.mean, self.std = mean, std
    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

# Calcular mean/std rápido para el ejemplo
vals = pd.read_csv('sunspots.csv')['Monthly Mean Total Sunspot Number'].astype('float32').values
norm = ZScore(vals.mean(), vals.std())

ts_dataset = TimeSeriesDataset('sunspots.csv', window=30, transform=norm)
ts_loader = DataLoader(ts_dataset, batch_size=32, shuffle=True)

xb, yb = next(iter(ts_loader))
print("Batch serie:", xb.shape, yb.shape)   

### Texto: la transformación es tokenizar y numericalizar

En texto, "transformar" significa convertir una cadena en una secuencia de IDs. Construimos un **vocabulario** (palabra → índice) y una función que numericaliza cada frase. Esa función es la transformación que va en `__getitem__`; su salida (secuencias de largo variable) es justo lo que luego junta el `collate_padding` de la sección 3.

In [ ]:
from collections import Counter
import re

frases = ["me gusta este curso",
          "el curso es excelente",
          "no me gusta esto",
          "excelente excelente curso"]
etiquetas = [1, 1, 0, 1]

# 1) construir vocabulario
contador = Counter(p for f in frases for p in f.split())
itos = ["<pad>", "<unk>"] + [p for p, _ in contador.most_common()]  # índice -> palabra
stoi = {p: i for i, p in enumerate(itos)}                           # palabra -> índice
print("Vocabulario:", stoi)

# 2) transformación: texto -> tensor de IDs
def numericalizar(texto):
    return torch.tensor([stoi.get(p, 1) for p in texto.split()], dtype=torch.long)  # 1 = <unk>

print("\n'me gusta este curso' ->", numericalizar("me gusta este curso"))

In [ ]:
class TextoConTransform(Dataset):
    def __init__(self, frases, etiquetas, transform):
        self.frases = frases; self.etiquetas = etiquetas; self.transform = transform
    def __len__(self): return len(self.frases)
    def __getitem__(self, i):
        return self.transform(self.frases[i]), self.etiquetas[i]

texto_ds = TextoConTransform(frases, etiquetas, transform=numericalizar)
# ¡y lo combinamos con el collate de padding de la sección 3!
loader = DataLoader(texto_ds, batch_size=4, collate_fn=collate_padding)
padded, longitudes, y = next(iter(loader))
print("Lote de texto ya numericalizado y con padding:\n", padded)
print("longitudes:", longitudes.tolist(), "| etiquetas:", y.tolist())

In [ ]:
def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z']+", text)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, path, seq_len=16, min_freq=1):
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()

        tokens = tokenize(text)
        counter = Counter(tokens)
        # Vocabulario: <pad>=0, <unk>=1, luego el resto
        vocab = {'<pad>': 0, '<unk>': 1}
        for tok, freq in counter.most_common():
            if freq >= min_freq:
                vocab[tok] = len(vocab)

        self.vocab = vocab
        self.ids = [vocab.get(t, vocab['<unk>']) for t in tokens]
        self.seq_len = seq_len

    def __len__(self):
        return len(self.ids) - self.seq_len

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.seq_len]
        y = self.ids[idx + 1 : idx + self.seq_len + 1]   # next-token
        return (torch.tensor(x, dtype=torch.long),
                torch.tensor(y, dtype=torch.long))

In [ ]:
text_dataset = TextDataset('sonnets.txt', seq_len=16)
print("Vocab size:", len(text_dataset.vocab))

text_loader = DataLoader(text_dataset, batch_size=8, shuffle=True)
xb, yb = next(iter(text_loader))
print("Batch texto:", xb.shape, yb.shape)